# DeepSplat SAM 3.1 GPU benchmark
Select an A100 or T4 GPU runtime. Add a Colab secret named HF_TOKEN with read access to facebook/sam3.1 and enable notebook access. Upload IMG_2132.MOV when prompted.

In [ ]:
!nvidia-smi
%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
%pip install -q 'git+https://github.com/facebookresearch/sam3.git@eef6839bcfc3458dd52ea80535dc6bd62aea86b1' 'numpy==1.26.4' 'opencv-python-headless==4.11.0.86' pycocotools einops psutil


In [ ]:
# If Colab asks you to restart after installation, restart the session before running this cell.
from google.colab import files, userdata
from huggingface_hub import hf_hub_download
from pathlib import Path
import cv2
from PIL import Image

checkpoint = hf_hub_download("facebook/sam3.1", "sam3.1_multiplex.pt", token=userdata.get("HF_TOKEN"))
if not Path("IMG_2132.MOV").exists():
    files.upload()
video = cv2.VideoCapture("IMG_2132.MOV")
frames = []
try:
    for index in range(121):
        ok, frame = video.read()
        if not ok:
            raise RuntimeError(f"Cannot decode frame {index}")
        if index in (0, 60, 120):
            frame = cv2.resize(frame, None, fx=.5, fy=.5)
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
finally:
    video.release()


In [ ]:
import statistics
import time
from unittest.mock import patch
import torch
from sam3.model_builder import build_sam3_multiplex_video_predictor

assert torch.cuda.is_available(), "Select a GPU runtime first"
print(torch.cuda.get_device_name())
autocast = torch.autocast

dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print(f'Autocast dtype: {dtype}')

def runtime_autocast(*args, **kwargs):
    if kwargs.get("dtype") == torch.bfloat16:
        kwargs["dtype"] = dtype
    return autocast(*args, **kwargs)

with patch.object(torch, "autocast", runtime_autocast):
    predictor = build_sam3_multiplex_video_predictor(
        checkpoint_path=checkpoint, use_fa3=False, use_rope_real=False
    )
try:
    model = predictor.model
    times = []
    with torch.inference_mode(), autocast("cuda", dtype=dtype):
        for index, image in zip((0, 60, 120), frames):
            torch.cuda.synchronize()
            start = time.perf_counter()
            state = model.init_state(resource_path=[image])
            _, output = model.add_prompt(state, frame_idx=0, text_str="peeling paint", output_prob_thresh=.5)
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - start
            print(f"Frame {index}: {elapsed:.3f}s, {len(output['out_probs'])} candidate masks")
            times.append(elapsed)
            del state, output
    print(f"Warm mean: {statistics.mean(times[1:]):.3f}s (model path only)")
finally:
    predictor.bf16_context.__exit__(None, None, None)
